# Step 8.5: Fine-tune the verifier (Colab A100)

Transfer learning: start from the DeBERTa-v3 NLI checkpoints the engine already uses and fine-tune on scientific / health fact-checking data (SciFact train, HealthVer, COVID-Fact).

**Runtime → Change runtime type → A100 GPU**, then run the cells in order (~45-60 min total).

Never used for training or model selection: SciFact dev (our benchmark) and HealthVer test.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
!git clone -q https://github.com/Anthony-Abumanneh/ai-truth-engine.git
%cd ai-truth-engine
!pip install -q -r requirements-train.txt

/content/ai-truth-engine


## 1. Data and starting models

In [3]:
!mkdir -p data/scifact && curl -sL https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz | tar -xz -C data/scifact
!python -m truth_engine.pipeline.nli download large
!python -m truth_engine.pipeline.nli download base
!python -m truth_engine.bench.training_data

saved to /content/ai-truth-engine/data/models/MoritzLaurer__DeBERTa-v3-large-mnli-fever-anli-ling-wanli
saved to /content/ai-truth-engine/data/models/MoritzLaurer__DeBERTa-v3-base-mnli-fever-anli
{
 "removed_dev_overlap": true,
 "train": {
  "n": 20094,
  "labels": {
   "neutral": 7583,
   "contradiction": 5973,
   "entailment": 6538
  },
  "sources": {
   "healthver": 10590,
   "covidfact": 4086,
   "scifact-negative": 2032,
   "scifact-nei": 1154,
   "scifact": 2232
  }
 },
 "val": {
  "n": 2197,
  "labels": {
   "neutral": 1164,
   "contradiction": 444,
   "entailment": 589
  },
  "sources": {
   "scifact-nei": 61,
   "scifact": 109,
   "scifact-negative": 110,
   "healthver": 1917
  }
 }
}


## 2. Before: the pretrained models

In [4]:
!TE_NLI=1 python scripts/eval_nli.py large base

large     scifact-dev     n=  321  accuracy 0.735  macro-F1 0.741  (12 ms/pair on cuda)
large     healthver-test  n= 1823  accuracy 0.515  macro-F1 0.455  (5 ms/pair on cuda)
base      scifact-dev     n=  321  accuracy 0.676  macro-F1 0.677  (3 ms/pair on cuda)
base      healthver-test  n= 1823  accuracy 0.502  macro-F1 0.441  (2 ms/pair on cuda)


## 3. Fine-tune (large ~25 min, base ~8 min on an A100)

In [5]:
!python scripts/finetune_verifier.py --size large

Loading weights: 100% 394/394 [00:00<00:00, 2141.20it/s]
100% 35/35 [00:06<00:00,  5.03it/s]
{'loss': '1.647', 'grad_norm': '31.24', 'learning_rate': '1.171e-06', 'epoch': '0.03981'}
{'loss': '1.204', 'grad_norm': '31.72', 'learning_rate': '2.367e-06', 'epoch': '0.07962'}
{'loss': '0.8585', 'grad_norm': '14.18', 'learning_rate': '3.562e-06', 'epoch': '0.1194'}
{'loss': '0.7845', 'grad_norm': '11.65', 'learning_rate': '4.757e-06', 'epoch': '0.1592'}
{'loss': '0.6438', 'grad_norm': '15.6', 'learning_rate': '5.952e-06', 'epoch': '0.199'}
{'loss': '0.6301', 'grad_norm': '22.8', 'learning_rate': '5.873e-06', 'epoch': '0.2389'}
{'loss': '0.5786', 'grad_norm': '16.36', 'learning_rate': '5.74e-06', 'epoch': '0.2787'}
{'loss': '0.5777', 'grad_norm': '13', 'learning_rate': '5.607e-06', 'epoch': '0.3185'}
{'loss': '0.5026', 'grad_norm': '11.38', 'learning_rate': '5.475e-06', 'epoch': '0.3583'}
{'loss': '0.4678', 'grad_norm': '6.642', 'learning_rate': '5.342e-06', 'epoch': '0.3981'}
{'loss': '0.47

In [6]:
!python scripts/finetune_verifier.py --size base

Loading weights: 100% 202/202 [00:00<00:00, 2290.72it/s]
100% 35/35 [00:03<00:00, 10.17it/s]
{'loss': '1.512', 'grad_norm': '19.96', 'learning_rate': '3.904e-06', 'epoch': '0.03981'}
{'loss': '1.064', 'grad_norm': '7.234', 'learning_rate': '7.888e-06', 'epoch': '0.07962'}
{'loss': '0.8508', 'grad_norm': '6.829', 'learning_rate': '1.187e-05', 'epoch': '0.1194'}
{'loss': '0.766', 'grad_norm': '11.22', 'learning_rate': '1.586e-05', 'epoch': '0.1592'}
{'loss': '0.6889', 'grad_norm': '10.21', 'learning_rate': '1.984e-05', 'epoch': '0.199'}
{'loss': '0.6635', 'grad_norm': '9.877', 'learning_rate': '1.958e-05', 'epoch': '0.2389'}
{'loss': '0.6166', 'grad_norm': '9.306', 'learning_rate': '1.913e-05', 'epoch': '0.2787'}
{'loss': '0.646', 'grad_norm': '7.317', 'learning_rate': '1.869e-05', 'epoch': '0.3185'}
{'loss': '0.5557', 'grad_norm': '7.631', 'learning_rate': '1.825e-05', 'epoch': '0.3583'}
{'loss': '0.5335', 'grad_norm': '7.403', 'learning_rate': '1.781e-05', 'epoch': '0.3981'}
{'loss': '

## 4. After: same evaluation

In [7]:
!TE_NLI=1 python scripts/eval_nli.py large ft-large base ft-base
!TE_NLI=1 TE_NLI_MODEL=ft-large python scripts/eval_stance_scifact.py dev

large     scifact-dev     n=  321  accuracy 0.735  macro-F1 0.741  (9 ms/pair on cuda)
large     healthver-test  n= 1823  accuracy 0.515  macro-F1 0.455  (5 ms/pair on cuda)
ft-large  scifact-dev     n=  321  accuracy 0.807  macro-F1 0.807  (7 ms/pair on cuda)
ft-large  healthver-test  n= 1823  accuracy 0.773  macro-F1 0.766  (5 ms/pair on cuda)
base      scifact-dev     n=  321  accuracy 0.676  macro-F1 0.677  (3 ms/pair on cuda)
base      healthver-test  n= 1823  accuracy 0.502  macro-F1 0.441  (2 ms/pair on cuda)
ft-base   scifact-dev     n=  321  accuracy 0.829  macro-F1 0.816  (3 ms/pair on cuda)
ft-base   healthver-test  n= 1823  accuracy 0.784  macro-F1 0.778  (2 ms/pair on cuda)
SciFact: 321 claim/evidence pairs {'neutral': 112, 'support': 138, 'contradict': 71}; model: local/truth-engine-verifier-large
(3.3s)
rules          accuracy 0.517   macro-F1 0.443   F1[support] 0.48  F1[contradict] 0.23  F1[neutral] 0.62
NLI model      accuracy 0.819   macro-F1 0.819   F1[support] 0.82

## 5. Save to Google Drive

Download `truth_engine_finetuned.zip` from Drive and unzip it inside the project folder on your Mac (it contains `data/models/local__...` and the eval results).

In [8]:
from google.colab import drive
drive.mount('/content/drive')
!zip -qr truth_engine_finetuned.zip data/models/local__truth-engine-verifier-large data/models/local__truth-engine-verifier-base data/finetune/eval_nli.json data/finetune/stats.json
!cp truth_engine_finetuned.zip /content/drive/MyDrive/ && ls -lh /content/drive/MyDrive/truth_engine_finetuned.zip

Mounted at /content/drive
-rw------- 1 root root 1.1G Oct 17 19:36 /content/drive/MyDrive/truth_engine_finetuned.zip
